# S³DT — Stage B: Projection Error Propagation (Django)

**Reviewer R1.2, part 2 — controlled propagation of claim-projection errors to downstream SIC and score decisions.**

This is a **standalone validation**, not a modification of the Django case-study notebook. It uses the existing 15 Django historical snapshots, the four fixed system-level SICs and the original fixed factor mapping/thresholds. It does not collect new incidents or change the published S³DT equations.

**Inputs:** Upload `TDSC_Django_System_SIC_Run.zip` (preferred) or `TDSC_Broader_Quantitative_Audit(2).zip`. Optionally upload the `S3DT_Django_StageA_Projection_Validation_Results(1).zip` for its scope and provenance. All three are already available from the earlier experiments.

**Important methodological boundary.** The current Django SIC scorer consumes the *generated* `risk_stage` field. Thus editing code-behaviour claims by itself does **not** propagate in the original implementation. This standalone experiment contains an explicitly documented evidence adapter that **recomputes the SIC decision from altered source-derived `code_behavior` claims** and applies the **unchanged** original structured-factor mapping, \(\lambda=0.5\), and classification cutoffs. Its semantic component compares the frozen policy text with the code-behaviour claims only; it does not include `risk_stage`. Results must be identified as a *controlled projection-error propagation evaluation*, not as an unchanged end-to-end detector run.

**Reference distinction:** Baseline agreement is measured against the 60 previously generated SIC outcomes (automated), **not 60 independent gold labels**. Historical pre/post fix windows are related and later fixes cannot be used as pre-fix detector inputs. The trial count is repeated perturbations, not independent benchmark samples.

No result is assumed before Colab executes the real MiniLM encoder.

In [ ]:
%pip -q install "sentence-transformers>=3,<6" pandas numpy


## 1 — Upload existing evidence

Run this once and select the Django system SIC results ZIP (or the broader quantitative audit ZIP). You may also select the Stage A validation ZIP in the same file picker. The files are saved to a local folder so subsequent cells do not depend on the upload dictionary remaining in memory.

In [ ]:
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/s3dt_stage_b_inputs')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
up = files.upload()
for filename, content in up.items():
    if not filename.lower().endswith('.zip'):
        raise ValueError('Please upload ZIP input files only.')
    dst = INPUT_DIR / Path(filename).name
    dst.write_bytes(content)
    print('Saved:', dst)

def find_input_zips(folder):
    import zipfile
    audit = stagea = None
    for candidate in sorted(folder.glob('*.zip')):
        with zipfile.ZipFile(candidate) as z:
            names = set(z.namelist())
        if ('system_sic_claim_inventory.csv' in names or 'system_sic_run/system_sic_claim_inventory.csv' in names) and \
                ('system_sic_transition_results.csv' in names or 'system_sic_run/system_sic_transition_results.csv' in names):
            # Prefer smaller, direct SIC ZIP over the large broader audit.
            if audit is None or candidate.stat().st_size < audit.stat().st_size:
                audit = candidate
        if 'projection_validation_report.json' in names:
            stagea = candidate
    return audit, stagea

AUDIT_ZIP, STAGE_A_ZIP = find_input_zips(INPUT_DIR)
if AUDIT_ZIP is None:
    raise FileNotFoundError('Upload TDSC_Django_System_SIC_Run.zip or a broader audit containing system_sic_run/.')
print('System SIC ZIP:', AUDIT_ZIP.name)
print('Stage A ZIP:', STAGE_A_ZIP.name if STAGE_A_ZIP else '(optional; not supplied)')

## 2 — Fixed evaluation protocol and claim-sensitive SIC adapter

The cell below is the complete implementation (no external `.py` file is required). For each of 60 original policy-checkpoint assessments, it reconstructs the status **without looking at `risk_stage`**, asserts agreement with the saved original SIC result, then injects one controlled error at a time into each applicable source-backed claim.

- **Omission:** delete one projected source fact; missing evidence becomes `UNKNOWN`.
- **Incorrect interpretation:** invert an existing security behaviour fact (source-unsupported counterfactual).
- **Spurious claim:** add a conflicting, unsupported fact, keeping the real one; conflicting evidence becomes `SUSPECT`.

The 60 recorded baseline decisions must all match before results are exported. The adapter's conflict/unknown handling is an explicit validation design choice, not claimed as behaviour previously measured in the original scorer.

In [ ]:
"""S3DT Stage B controlled projection-error propagation. Not the unchanged detector.
The source-claim adapter re-evaluates the same system SIC checks after the claims are changed.
"""
import csv, hashlib, io, json, math, random, re, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

POLICY_EXPECTED = {
 'DJANGO-AUTHZ-001':{
  'InlineModelAdmin.has_add_permission':'change_permission_controls_mutation',
  'InlineModelAdmin.has_change_permission':'change_permission_controls_mutation',
  'InlineModelAdmin.has_delete_permission':'change_permission_controls_mutation'},
 'DJANGO-URL-001':{'AdminURLFieldWidget.get_context':'scheme_validation_present',
                   'admin/widgets/url.html':'url_valid_gate'},
 'DJANGO-UPLOAD-001':{'Storage.get_available_name':'path_validation_present',
                      'Storage.generate_filename':'path_validation_present',
                      'UploadedFile._set_name':'path_validation_present'},
 'DJANGO-TEMPLATE-001':{'TemplateDetailView.get_context_data':'safe_join_path'}
}
UNSAFE = {'change_permission_controls_mutation':'view_permission_controls_mutation',
          'view_permission_controls_mutation':'change_permission_controls_mutation',
          'scheme_validation_present':'scheme_validation_not_identified',
          'scheme_validation_not_identified':'scheme_validation_present',
          'url_valid_gate':'url_valid_gate_missing',
          'url_valid_gate_missing':'url_valid_gate',
          'path_validation_present':'path_validation_not_identified',
          'path_validation_not_identified':'path_validation_present',
          'safe_join_path':'direct_untrusted_path_join',
          'direct_untrusted_path_join':'safe_join_path'}


def derive_status(policy_id, facts):
    """Documented evidence adapter for the policy families in the original Django code.
    Conflicts and missing source facts are unresolved, never silently marked SATISFIED.
    URL checks follow the original conservative guard logic; unknown ≠ PASS.
    """
    needed=POLICY_EXPECTED[policy_id]
    grouped=defaultdict(set)
    for f in facts:
        if f['subject'] in needed:
            grouped[f['subject']].add(f['object'])
    if not all(grouped.get(s) for s in needed):return 'UNKNOWN'
    if any(len(grouped[s])>1 or not grouped[s].issubset({needed[s],UNSAFE[needed[s]]}) for s in needed):return 'SUSPECT'
    if all(next(iter(grouped[s]))==needed[s] for s in needed):return 'SATISFIED'
    if policy_id=='DJANGO-AUTHZ-001':
        return 'VIOLATION' if any(next(iter(grouped[s]))=='view_permission_controls_mutation' for s in needed) else 'SUSPECT'
    if policy_id=='DJANGO-URL-001':
        url=next(iter(grouped['AdminURLFieldWidget.get_context']))
        tmpl=next(iter(grouped['admin/widgets/url.html']))
        return 'VIOLATION' if url=='scheme_validation_not_identified' and tmpl=='url_valid_gate_missing' else 'SUSPECT'
    if policy_id=='DJANGO-UPLOAD-001':
        return 'VIOLATION' if any(next(iter(grouped[s]))=='path_validation_not_identified' for s in needed) else 'SUSPECT'
    if policy_id=='DJANGO-TEMPLATE-001':
        return 'VIOLATION' if next(iter(grouped['TemplateDetailView.get_context_data']))=='direct_untrusted_path_join' else 'SUSPECT'
    raise KeyError(policy_id)


def structured(status,config):
    factors=config['scoring']['factor_map'].get(status)
    if factors is None:return float('nan')
    return sum(config['scoring']['factor_weights'][k]*factors[k] for k in ('C','O','H'))


def semantic_summary(policy_id,facts,config):
    policy=next(p for p in config['policies'] if p['id']==policy_id)
    # Excludes risk_stage, label, advisory, fixes, and activity-derived outcomes.
    return ' '.join(sorted(
        f"{x['subject'].replace('_',' ')}: {x['object'].replace('_',' ')}." for x in facts))


def inject(facts,kind,target):
    new=[dict(f) for f in facts]
    loc=next((i for i,f in enumerate(new) if f['subject']==target),None)
    if loc is None:raise ValueError('Missing target '+target)
    prior=new[loc]['object']
    opposite=UNSAFE.get(prior)
    if opposite is None:raise ValueError('No controlled inversion for '+prior)
    if kind=='omission':new.pop(loc)
    elif kind=='incorrect_interpretation':new[loc]['object']=opposite
    elif kind=='spurious_claim':
        new.append({**dict(new[loc]),'object':opposite,'injection_type':'unsupported_conflicting_claim'})
    else:raise ValueError(kind)
    return new,prior,opposite


def score_label(d,config):
    if not math.isfinite(d):return 'UNKNOWN'
    cut=config['scoring']['score_only_cutoffs']
    return 'PASS' if d<=cut['pass'] else ('VIOLATION' if d>=cut['violation'] else 'SUSPECT')


def load_inputs(audit_path,stagea_path=None):
    z=zipfile.ZipFile(audit_path)
    prefix='system_sic_run/' if 'system_sic_run/system_sic_claim_inventory.csv' in z.namelist() else ''
    def get(name):
        match=prefix+name
        if match not in z.namelist():raise FileNotFoundError(match+' absent from '+str(audit_path))
        return z.read(match)
    claims=pd.read_csv(io.BytesIO(get('system_sic_claim_inventory.csv'))).fillna('')
    old=pd.read_csv(io.BytesIO(get('system_sic_transition_results.csv'))).fillna('')
    config=json.loads(get('predeclared_django_system_sics.json'))
    assert set(POLICY_EXPECTED)=={x['id'] for x in config['policies']}
    stage_summary={}
    if stagea_path:
        stagez=zipfile.ZipFile(stagea_path)
        if 'projection_validation_report.json' not in stagez.namelist():raise ValueError('Invalid Stage A validation ZIP')
        stage_summary=json.loads(stagez.read('projection_validation_report.json'))
    return claims,old,config,stage_summary


def run(audit_path,out_dir,stagea_path=None,encoder=None):
    """Compute baseline fidelity, fixed perturbation pairs, and reporting CSVs.
    encoder: SentenceTransformer-like .encode([...],normalize_embeddings=True,show_progress_bar=False)
    """
    if encoder is None:raise RuntimeError('Real MiniLM encoder required; no substitute values can be published.')
    out_dir=Path(out_dir);out_dir.mkdir(parents=True,exist_ok=True)
    claims,old,config,stagea=load_inputs(audit_path,stagea_path)
    if config['scoring']['lambda_p']!=0.5:raise ValueError('Unexpected fusion parameter: use original fixed config')
    selected=claims[claims.predicate=='code_behavior'].copy()
    keys=['case_id','checkpoint','commit','policy_id']
    case_rows=[];invalid=[];batch=[];summaries=[]
    for key, group in selected.groupby(keys,sort=True):
        case,checkpoint,commit,policy=key
        f=group[['subject','object','evidence']].to_dict('records')
        base=derive_status(policy,f)
        expected=old[(old.case_id==case)&(old.checkpoint==checkpoint)&(old.sic_id==policy)]
        if len(expected)!=1:raise RuntimeError('Checkpoint mapping not unique '+str(key))
        saved=expected.iloc[0]
        if base != saved.sic_outcome:
            invalid.append({'case_id':case,'checkpoint':checkpoint,'policy_id':policy,'saved_status':saved.sic_outcome,'reconstructed_status':base})
        is_ref=str(saved.is_reference_only).lower()=='true'
        for target in sorted(POLICY_EXPECTED[policy]):
            for typ in ['omission','incorrect_interpretation','spurious_claim']:
                altered,original,injected=inject(f,typ,target)
                result=derive_status(policy,altered)
                record={'case_id':case,'checkpoint':checkpoint,'commit':commit,'policy_id':policy,
                  'target_subject':target,'error_type':typ,'source_object':original,'injected_object':injected,
                  'baseline_status':base,'altered_status':result,'reference_only':is_ref,
                  'baseline_fact_count':len(f),'altered_fact_count':len(altered),
                  'same_sic':result==base,'baseline_structured':structured(base,config),
                  'altered_structured':structured(result,config),
                  'source_provenance':group[group.subject==target].iloc[0].evidence,
                  'injected_evidence_not_grounded':typ=='spurious_claim',
                  'context':'CONTROLLED_PERTURBATION_NOT_REAL_EXTRACTION_ERROR'}
                batch.append((record,semantic_summary(policy,f,config),semantic_summary(policy,altered,config)))
                summaries.extend([batch[-1][1],batch[-1][2]])
    if invalid:
        pd.DataFrame(invalid).to_csv(out_dir/'baseline_reconstruction_failures.csv',index=False)
        raise AssertionError(f'{len(invalid)} baseline statuses differ; no propagation scores produced. Inspect baseline_reconstruction_failures.csv')
    # Embeddings are re-computed from injected source behaviour claims (no generated risk_stage leakage).
    policies={x['id']:x['description'] for x in config['policies']}
    uniq=list(dict.fromkeys(summaries+list(policies.values())))
    v=np.asarray(encoder.encode(uniq,normalize_embeddings=True,show_progress_bar=False),dtype=float)
    if len(v)!=len(uniq):raise RuntimeError('Encoder returned wrong number of vectors')
    lookup={t:v[i] for i,t in enumerate(uniq)}
    lam=config['scoring']['lambda_p']
    output=[]
    for r,original,altered in batch:
        # Policy-to-source embedding divergence, recomputed without generated risk_stage.
        policy_vec=lookup[policies[r['policy_id']]]
        d_original=float(np.clip(1-np.dot(policy_vec,lookup[original]),0,1))
        d_altered=float(np.clip(1-np.dot(policy_vec,lookup[altered]),0,1))
        change=float(d_altered-d_original)
        r['baseline_semantic_drift']=d_original
        r['altered_semantic_drift']=d_altered
        r['semantic_change_due_to_error']=change
        r['delta_structured']= (r['altered_structured']-r['baseline_structured']) if math.isfinite(r['altered_structured']) else np.nan
        r['baseline_hybrid']=lam*r['baseline_structured']+(1-lam)*d_original if math.isfinite(r['baseline_structured']) else np.nan
        r['altered_hybrid']=lam*r['altered_structured']+(1-lam)*d_altered if math.isfinite(r['altered_structured']) else np.nan
        r['hybrid_score_delta']=r['altered_hybrid']-r['baseline_hybrid'] if math.isfinite(r['altered_hybrid']) else np.nan
        r['baseline_hybrid_label']=score_label(r['baseline_hybrid'],config)
        r['altered_hybrid_label']=score_label(r['altered_hybrid'],config)
        r['hybrid_label_changed']=r['baseline_hybrid_label']!=r['altered_hybrid_label']
        r['decision_changed']=r['baseline_status']!=r['altered_status']
        r['changed_to_unknown']=r['altered_status']=='UNKNOWN'
        r['changed_to_suspect']=r['altered_status']=='SUSPECT' and r['baseline_status']!='SUSPECT'
        r['violation_lost']=r['baseline_status']=='VIOLATION' and r['altered_status']!='VIOLATION'
        r['false_violation_new']=r['baseline_status']=='SATISFIED' and r['altered_status']=='VIOLATION'
        r['new_alert_from_satisfied']=r['baseline_status']=='SATISFIED' and r['altered_status'] in ('SUSPECT','VIOLATION')
        r['baseline_structured_label']=score_label(r['baseline_structured'],config)
        r['altered_structured_label']=score_label(r['altered_structured'],config)
        output.append(r)
    result=pd.DataFrame(output)
    result.to_csv(out_dir/'stage_b_all_controlled_perturbations.csv',index=False)
    focus=result[result.checkpoint.isin(['pre_fix_detector_state','post_fix_reference_only'])]
    focus.to_csv(out_dir/'stage_b_advisory_paired_states.csv',index=False)
    grouped=result.groupby('error_type').agg(n=('decision_changed','size'),decision_changes=('decision_changed','sum'),unknown=('changed_to_unknown','sum'),suspect_escalations=('changed_to_suspect','sum'),lost_violations=('violation_lost','sum'),new_false_violation=('false_violation_new','sum'),new_alerts=('new_alert_from_satisfied','sum'),mean_semantic_change=('semantic_change_due_to_error','mean'),hybrid_label_changes=('hybrid_label_changed','sum')).reset_index()
    grouped['decision_change_rate']=grouped.decision_changes/grouped.n
    grouped.to_csv(out_dir/'stage_b_per_error_type.csv',index=False)
    focused=focus.groupby(['policy_id','error_type']).agg(n=('decision_changed','size'),decision_changes=('decision_changed','sum'),lost_violations=('violation_lost','sum'),new_alerts=('new_alert_from_satisfied','sum')).reset_index()
    focused.to_csv(out_dir/'stage_b_by_policy_advisory_states.csv',index=False)
    # Retain caveats on correlation and truth separation.
    metadata={'stage':'B controlled injection at code_behavior claim layer',
      'source':'Historical Django 15 checkpoints x four system SICs',
      'stage_a_provided':bool(stagea_path),'stage_a_report':stagea,
      'primary_estimand':'Decision change vs UNPERTURBED SYSTEM DECISION, not externally verified detection accuracy',
      'source_baseline_agreement':{'matched':int(len(old)),'total':int(len(old))},
      'n_perturbations':len(result),'n_advisory_state_perturbations':len(focus),
      'original_detector_unchanged':True,'separate_evidence_adapter':True,
      'adapter_limitation':'Re-derives SIC status from injected source-derived facts because released Django system_evaluate_constraint reads generated risk_stage. NOT identical to unmodified detector runtime.',
      'embedding_limitation':'Compares frozen system-policy description against projected code-behaviour facts, excluding risk_stage; this is a Stage-B source-fact adapter, not a replay of the original policy+claim summary embedding formulation.',
      'labels_limitation':'Unperturbed SIC outcome is an automated reference for consistency, not independent security ground truth. Historical fix states are evaluation-only references.',
      'uncertainty':'Perturbations are deterministic and clustered by history, snapshot, policy and claim; no independent-sample confidence intervals should be inferred.',
      'config_status':config.get('status'),'frozen_lambda':lam,'source_zip_sha256':hashlib.sha256(Path(audit_path).read_bytes()).hexdigest(),
      'stage_a_zip_sha256':hashlib.sha256(Path(stagea_path).read_bytes()).hexdigest() if stagea_path else None,
      'encoder':'sentence-transformers/all-MiniLM-L6-v2'}
    (out_dir/'stage_b_protocol_and_limitations.json').write_text(json.dumps(metadata,indent=2,default=str))
    (out_dir/'README.txt').write_text('Stage B: controlled claim perturbations, reproducing the unmodified 60-row SIC assessment baseline.\n'+
      'Use stage_b_per_error_type.csv and stage_b_by_policy_advisory_states.csv for descriptive decision stability.\n'+
      'Stage B hybrid scores use the unchanged lambda and cutoffs, but DIFFERENT source-fact-only embedding summaries, so cannot be interpreted as the original published detector scores.\n'+
      'Original case-study notebook, historical evidence and SIC parameters were not edited.\n')
    pack=out_dir.parent/'S3DT_Django_StageB_Error_Propagation_Results.zip'
    with zipfile.ZipFile(pack,'w',zipfile.ZIP_DEFLATED) as z:
        for f in sorted(out_dir.iterdir()):
            if f.is_file():z.write(f,arcname=f.name)
    return {'archive':str(pack),'summaries':grouped.to_dict('records'),'rows':len(result),'baseline_matched':len(old)}


## 3 — Run Stage B with the real MiniLM encoder

This cell performs the actual embedding computations and exports both a summary and claim-level CSV. Outputs include per-type decision-change rates, lost violations, false alerts and changes in source-fact-only hybrid scores. **Unknown outcomes are not treated as PASS.**

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
from IPython.display import display

# Avoid relying on variable names from the original master notebook.
if 'AUDIT_ZIP' not in globals() or AUDIT_ZIP is None:
    AUDIT_ZIP, STAGE_A_ZIP = find_input_zips(Path('/content/s3dt_stage_b_inputs'))
if AUDIT_ZIP is None:
    raise FileNotFoundError('Missing uploaded Django system SIC ZIP. Run Section 1.')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
OUTPUT_DIR = Path('/content/s3dt_stage_b_outputs')
result = run(AUDIT_ZIP, OUTPUT_DIR, STAGE_A_ZIP, encoder=model)
print('Reconstructed original SIC outcomes:', result['baseline_matched'], '/ 60')
print('Controlled claim-perturbation assessments:', result['rows'])
display(pd.read_csv(OUTPUT_DIR/'stage_b_per_error_type.csv'))
print('Saved results ZIP:', result['archive'])

## 4 — Export and download (can be rerun without recomputing embeddings)

The ZIP contains the complete paired results, summaries, a protocol and limitations JSON, plus README. It uses absolute paths and avoids the earlier `S15_OUT`/`S15_SOURCE` variable errors.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

output_dir = Path('/content/s3dt_stage_b_outputs')
required = ['stage_b_all_controlled_perturbations.csv',
            'stage_b_per_error_type.csv',
            'stage_b_protocol_and_limitations.json']
missing = [name for name in required if not (output_dir/name).is_file()]
if missing:
    raise FileNotFoundError(f'Outputs missing {missing}; run Section 3 first.')
zip_path = Path('/content/S3DT_Django_StageB_Error_Propagation_Results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for item in sorted(output_dir.iterdir()):
        if item.is_file():
            z.write(item,arcname=item.name)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
files.download(str(zip_path))

## Interpretation for the TDSC revision

Report Stage A's claim-level extraction accuracy separately. Stage B measures **how errors introduced into the claim representation perturb the SIC decision**, conditional on this Django source-fact adapter. Distinguish automated baseline decision stability from externally verified true positive/false negative rates.

Do **not** call the resulting perturbations independent observations or treat unknown decisions as benign. The source-fact-only MiniLM channel and derived scores are explicitly validation-specific: they do not replace the original full case-study hybrid values. The 4 vulnerability/fix pairs and 7 ordinary controls are repeated historical source states, not a random sample of development projects.

One concise main-paper table can report decision-change rates by perturbation type; preserve full per-policy, per-state results and reproducibility details in supplementary material/GitHub.